In [1]:
from dotenv import load_dotenv
import os

load_dotenv('../.env')  # path relative to notebooks/ folder
API_KEY = os.getenv('API_KEY')


In [ ]:
import requests

url = "http://127.0.0.1:5000/predict"

headers = {"X-API-Key": API_KEY}

data = [{
    "step": 1,
    "type": 3, 
    "amount": 4878.0,
    "oldbalanceOrg": 170136.0,
    "newbalanceOrig": 165258.0,
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
    "isFlaggedFraud": 0
}]

response = requests.post(url, json=data, headers=headers)
print(response.json())

{'fraud_probability': 0.0, 'label': 'LEGITIMATE', 'prediction': 0}


In [5]:
import pandas as pd, numpy as np, joblib, warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE

print('Loading data...')
df = pd.read_csv('../data/Synthetic_Financial_datasets_log.csv')
df = df.dropna().drop(columns=['nameOrig','nameDest'])
df['type'] = df['type'].astype('category').cat.codes

# Engineer new features
df['balance_error_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig'] - df['amount']
df['balance_error_dest'] = df['newbalanceDest'] - df['oldbalanceDest'] - df['amount']

X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
num_cols = ['amount','oldbalanceOrg','oldbalanceDest','step','balance_error_orig','balance_error_dest']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

model = XGBClassifier(subsample=1.0, reg_lambda=1, reg_alpha=0.1,
    n_estimators=300, max_depth=7, learning_rate=0.2,
    gamma=0.5, colsample_bytree=0.8, random_state=42, eval_metric='logloss')

model.fit(X_train_sm, y_train_sm)
y_prob = model.predict_proba(X_test)[:,1]
print(f'New ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, (y_prob>=0.6835).astype(int)))

Loading data...
New ROC-AUC: 0.9996
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.87      0.99      0.93      1643

    accuracy                           1.00   1272524
   macro avg       0.94      0.99      0.96   1272524
weighted avg       1.00      1.00      1.00   1272524

